# crewai.Knowledge




`Knowledge` is a collection of sources and setup for the vector store to save and query relevant context.
* `BaseKnowledgeSource`s (see [_KNOWN_SOURCES](https://github.com/crewAIInc/crewAI/blob/1.15.16/lib/crewai/src/crewai/knowledge/knowledge.py#L23-L31))
* `EmbedderConfig`
* `BaseKnowledgeStorage`

Key benefits of using [Knowledge](https://docs.crewai.com/v1.15.16/en/concepts/knowledge) in crewAI:
* Enhance agents with domain-specific information
* Support decisions with real-world data
* Maintain context across conversations
* Ground responses in factual information

`Knowledge` is a named entity (by `collection_name`).


`Knowledge` is a [Pydantic model](https://pydantic.dev/docs/validation/latest/concepts/models/).

<br>

```py
class Knowledge(BaseModel)
```


## class_method_map

[Among the tracked classes by MLflow](https://github.com/mlflow/mlflow/blob/v3.15.1/mlflow/crewai/__init__.py#L78).

<br>

```py
class_method_map.update({"crewai.Knowledge": ["query"]})
```

## SpanType.RETRIEVER

`crewai.Knowledge` is assigned [SpanType.RETRIEVER](https://github.com/mlflow/mlflow/blob/v3.15.1/mlflow/crewai/autolog.py#L242-L243) span type in MLflow.

## _get_span_type

[_get_span_type](https://github.com/mlflow/mlflow/blob/v3.15.1/mlflow/crewai/autolog.py#L201-L247) assigns span types to crewAI instances.

crewAI's Instance | Span Type
-|-
Agent | `AGENT`
BaseAgentExecutor | `MEMORY`
Crew | `CHAIN`
`EntityMemory` | `MEMORY`
Flow | `CHAIN`
Knowledge | `RETRIEVER`
LLM | `LLM`
`LongTermMemory` | `MEMORY`
`ShortTermMemory` | `MEMORY`
Task | `CHAIN`

# 🛠️ Set Up Environment

In [0]:
%pip install -qU "mlflow-skinny[databricks]>=3.15.1" "crewai[tools]>=1.15.16" "databricks-langchain"
%restart_python

In [0]:
%pip show crewai

In [0]:
%pip show mlflow-skinny

# crewai/knowledge/knowledge.py


`crewai.Knowledge` is a class defined in `crewai/knowledge/knowledge.py`.

In [0]:
from crewai.knowledge.knowledge import Knowledge


`crewai.Knowledge` is re-exported in `crewai/__init__.py` for simplicity (and publicity = the official API).

In [0]:
from crewai import Knowledge

In [0]:
import mlflow

mlflow.crewai.autolog()


`Crew`s and `Agent`s can have dedicated `Knowledge`s.

* `Crew.knowledge_sources` - Knowledge sources for the crew.
* `BaseAgent.knowledge_sources` - Knowledge sources for the agent.

# StringKnowledgeSource

`StringKnowledgeSource` is a knowledge source (`BaseKnowledgeSource`) that stores and queries plain text content using embeddings.

`StringKnowledgeSource` only accepts string content.

In [0]:
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource

# Create knowledge source with user preferences
content = "Users name is John. He is 30 years old and lives in San Francisco."
string_source = StringKnowledgeSource(
    content=content, 
    metadata={"preference": "personal"}
)
print(string_source)

In [0]:
assert string_source.storage is None


# Knowledge Storage


`BaseKnowledgeSource` should have a `BaseKnowledgeStorage` defined before use.

`KnowledgeStorage` is the only built-in `BaseKnowledgeStorage` in crewAI for embeddings for memory entries.

In [0]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage

storage = KnowledgeStorage(collection_name="docs")
print(storage)

In [0]:
assert storage.embedder is None

In [0]:
# https://docs.langchain.com/oss/python/integrations/embeddings/databricks

from databricks_langchain import DatabricksEmbeddings

# Use Databricks-native embedding model
databricks_embeddings = DatabricksEmbeddings(
    endpoint="databricks-gte-large-en",
    # Specify parameters for embedding queries and documents if needed
    # query_params={...},
    # document_params={...},
)
print(databricks_embeddings)

In [0]:
from chromadb.utils.embedding_functions.chroma_langchain_embedding_function import create_langchain_embedding

# Create a chromadb-compatible embedding function
embedding_function = create_langchain_embedding(databricks_embeddings)

In [0]:
# Set the embedder config on the storage
storage.embedder = {
    "provider": "custom",
    "config": {
        "embedding_function": embedding_function
    }
}
print(f"Storage embedder: {storage.embedder}")

In [0]:
from crewai.rag.chromadb.config import ChromaDBConfig

chroma_config = ChromaDBConfig(embedding_function=embedding_function)
print(chroma_config)

In [0]:
from crewai.rag.config.utils import set_rag_config

set_rag_config(chroma_config)

In [0]:
storage.search(query=list("search test"))

In [0]:
string_source.storage = storage

In [0]:
print(string_source)

In [0]:
# Reset the stale collection that has a conflicting persisted "openai" embedding function
storage.reset()
string_source.add()

In [0]:
from chromadb.utils.embedding_functions.chroma_langchain_embedding_function import ChromaLangchainEmbeddingFunction

# Fix: chromadb calls embed_query(input=...) but ChromaLangchainEmbeddingFunction expects 'text'
_orig_embed_query = ChromaLangchainEmbeddingFunction.embed_query

def _patched_embed_query(self, text=None, **kwargs):
    if text is None:
        text = kwargs.pop('input', None)
    if isinstance(text, list):
        return self.embedding_function.embed_documents(text)
    return _orig_embed_query(self, text)

ChromaLangchainEmbeddingFunction.embed_query = _patched_embed_query

storage.search(query=list("search test"))